# Thực hành Data Cleaning với Python

Notebook này được thiết kế để thực hành các nội dung chính của bài **Data Cleaning in Python**.

## Mục tiêu

Sau khi hoàn thành notebook, bạn có thể:

- kiểm tra cấu trúc và chất lượng dữ liệu;
- phát hiện và xử lý duplicate records;
- xác định numerical và categorical variables;
- kiểm tra unique values;
- chuẩn hóa text và tên cột;
- chuyển đổi dữ liệu ngày tháng;
- phát hiện và xử lý missing values;
- phát hiện potential outliers bằng IQR;
- kiểm tra các giá trị ngoài phạm vi hợp lý;
- kiểm tra business rules;
- validation dữ liệu sau khi làm sạch.

> **Cách làm:** Đọc phần nhắc lại lý thuyết, chạy các ví dụ mẫu, sau đó hoàn thiện các ô có chữ `TODO`.


## 0. Chuẩn bị môi trường

### Nhắc lại lý thuyết

Trong Data Cleaning, ba thư viện thường được sử dụng là:

- `pandas`: đọc, kiểm tra và biến đổi dữ liệu dạng bảng;
- `numpy`: xử lý dữ liệu số và missing values;
- `matplotlib`: trực quan hóa phân phối và outliers.

Chạy ô dưới đây trước khi bắt đầu.


In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)


## 1. Tạo dữ liệu thực hành

Trong thực tế, dữ liệu có thể đến từ CSV, Excel, database hoặc API.  
Notebook này tạo sẵn một dataset bán hàng có chủ ý chứa nhiều lỗi để bạn thực hành.

Các cột gồm:

- `Order ID`
- `Customer ID`
- `Region`
- `Order Date`
- `Product`
- `Quantity`
- `Unit Price`
- `Revenue`
- `Payment Method`
- `Weight`
- `Weight Unit`


In [6]:
raw_data = {
    "Order ID": [
        "O001", "O002", "O003", "O004", "O005",
        "O006", "O007", "O008", "O009", "O010",
        "O011", "O012", "O013", "O014", "O015",
        "O005"   # duplicate Order ID
    ],
    "Customer ID": [
        "C001", "C002", "C003", "C004", "C005",
        "C006", "C007", "C008", "C009", "C010",
        "C011", "C012", "C013", "C014", "C015",
        "C005"
    ],
    "Region": [
        " Hanoi ", "HANOI", "ha noi", "Ho Chi Minh", "HO CHI MINH ",
        "Da Nang", "da nang", "Hanoi", "Ho Chi Minh", " Ha Noi",
        "DA NANG", "Hanoi", np.nan, "Ho Chi Minh", "Hanoi",
        "HO CHI MINH "
    ],
    "Order Date": [
        "2026-09-01", "02/09/2026", "Sep 3, 2026", "2026-09-04", "05/09/2026",
        "2026-09-06", "07/09/2026", "2026-09-08", "09/09/2026", "2026-09-10",
        "Sep 11, 2026", "2026-09-12", "13/09/2026", "invalid-date", "2026-09-15",
        "05/09/2026"
    ],
    "Product": [
        "Coffee", "Tea", "Coffee", "Juice", "Tea",
        "Coffee", "Juice", "Tea", "Coffee", "Juice",
        "Tea", "Coffee", "Juice", "Tea", "Coffee",
        "Tea"
    ],
    "Quantity": [
        2, 3, 1, 4, 2,
        5, 3, 2, -1, 4,
        100, 2, 3, 1, 2,
        2
    ],
    "Unit Price": [
        50_000, 40_000, 50_000, 60_000, np.nan,
        50_000, 60_000, 40_000, 50_000, 60_000,
        40_000, 50_000, np.nan, 40_000, 50_000,
        np.nan
    ],
    "Revenue": [
        100_000, 120_000, 50_000, 240_000, 80_000,
        250_000, 180_000, 80_000, -50_000, 240_000,
        4_000_000, 100_000, 180_000, 40_000, 999_999,
        80_000
    ],
    "Payment Method": [
        " Cash", "cash", "CASH", "Credit Card", "credit card ",
        "Bank Transfer", "bank transfer", "Cash", "cash", "Credit Card",
        "CASH", "Bank Transfer", np.nan, "credit card", "Cash",
        "credit card "
    ],
    "Weight": [
        1.2, 800, 1.5, 2.0, 650,
        1.0, 900, 0.8, 1.1, 750,
        2.5, 1.4, 600, 1.0, 1.3,
        650
    ],
    "Weight Unit": [
        "kg", "g", "kg", "kg", "g",
        "kg", "g", "kg", "kg", "g",
        "kg", "kg", "g", "kg", "kg",
        "g"
    ]
}

df = pd.DataFrame(raw_data)
df


,Order ID,Customer ID,Region,Order Date,Product,Quantity,Unit Price,Revenue,Payment Method,Weight,Weight Unit
0,O001,C001,Hanoi,2026-09-01,Coffee,2,50000.0,100000,Cash,1.2,kg
1,O002,C002,HANOI,02/09/2026,Tea,3,40000.0,120000,cash,800.0,g
2,O003,C003,ha noi,"Sep 3, 2026",Coffee,1,50000.0,50000,CASH,1.5,kg
3,O004,C004,Ho Chi Minh,2026-09-04,Juice,4,60000.0,240000,Credit Card,2.0,kg
4,O005,C005,HO CHI MINH,05/09/2026,Tea,2,NaN,80000,credit card,650.0,g
5,O006,C006,Da Nang,2026-09-06,Coffee,5,50000.0,250000,Bank Transfer,1.0,kg
6,O007,C007,da nang,07/09/2026,Juice,3,60000.0,180000,bank transfer,900.0,g
7,O008,C008,Hanoi,2026-09-08,Tea,2,40000.0,80000,Cash,0.8,kg
8,O009,C009,Ho Chi Minh,09/09/2026,Coffee,-1,50000.0,-50000,cash,1.1,kg
9,O010,C010,Ha Noi,2026-09-10,Juice,4,60000.0,240000,Credit Card,750.0,g


# 2. Đánh giá chất lượng dữ liệu ban đầu

### Nhắc lại lý thuyết

Trước khi sửa dữ liệu, cần trả lời:

- Dataset có bao nhiêu dòng và cột?
- Kiểu dữ liệu hiện tại là gì?
- Có missing values không?
- Có duplicate records không?
- Numerical và categorical variables gồm những cột nào?

Các lệnh thường dùng:

```python
df.shape
df.head()
df.info()
df.isna().sum()
df.duplicated().sum()
```


In [7]:
# Ví dụ: xem 5 dòng đầu
df.head()


,Order ID,Customer ID,Region,Order Date,Product,Quantity,Unit Price,Revenue,Payment Method,Weight,Weight Unit
0,O001,C001,Hanoi,2026-09-01,Coffee,2,50000.0,100000,Cash,1.2,kg
1,O002,C002,HANOI,02/09/2026,Tea,3,40000.0,120000,cash,800.0,g
2,O003,C003,ha noi,"Sep 3, 2026",Coffee,1,50000.0,50000,CASH,1.5,kg
3,O004,C004,Ho Chi Minh,2026-09-04,Juice,4,60000.0,240000,Credit Card,2.0,kg
4,O005,C005,HO CHI MINH,05/09/2026,Tea,2,NaN,80000,credit card,650.0,g


In [10]:
# TODO 1:
# 1. In kích thước dataset
# 2. Xem thông tin bằng info()
# 3. Đếm missing values
# 4. Đếm duplicate rows

print("Shape:", df.shape)

# df.________()

# df.shape()

print("\nMissing values:")
print(df.isna().sum())

print("\nDuplicate rows:", df.duplicated().sum())


Shape: (16, 11)

Missing values:
Order ID          0
Customer ID       0
Region            1
Order Date        0
Product           0
Quantity          0
Unit Price        3
Revenue           0
Payment Method    1
Weight            0
Weight Unit       0
dtype: int64

Duplicate rows: 1


### Câu hỏi suy nghĩ

1. Cột nào đang có missing values?
2. `Order Date` hiện có kiểu dữ liệu gì?
3. Có thể kết luận ngay một cột số là numerical feature không? Vì sao?


# 3. Chuẩn hóa tên cột

### Nhắc lại lý thuyết

Tên cột nhất quán giúp code dễ đọc và ít lỗi hơn.

Một quy tắc phổ biến:

```text
"Order ID"      → "order_id"
"Payment Method"→ "payment_method"
```

Có thể dùng:

```python
.str.strip()
.str.lower()
.str.replace()
```


In [ ]:
# Ví dụ
example_columns = pd.Index([" Customer Name ", "Order Value", "Region-Code"])

(
    example_columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace("-", "_")
)


In [11]:
# TODO 2:
# Chuẩn hóa toàn bộ tên cột của df
# Gợi ý: strip -> lower -> replace space -> replace hyphen

df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace("-", "_")
)

df.columns


Index(['order_id', 'customer_id', 'region', 'order_date', 'product',
       'quantity', 'unit_price', 'revenue', 'payment_method', 'weight',
       'weight_unit'],
      dtype='str')

# 4. Kiểm tra và xử lý Duplicate Records

### Nhắc lại lý thuyết

`duplicated()` trả về `True` cho các dòng bị xem là trùng.

```python
df.duplicated()
df.duplicated().sum()
```

Tuy nhiên, duplicate phải được đánh giá theo **ý nghĩa của dữ liệu**.

Trong dữ liệu giao dịch, hai dòng giống nhau có thể vẫn là hai giao dịch hợp lệ nếu chúng có `order_id` khác nhau.

Ở đây, `order_id` được xem là khóa nghiệp vụ quan trọng.


In [ ]:
# Kiểm tra Order ID bị lặp
df[df.duplicated(subset=["order_id"], keep=False)]


In [ ]:
# TODO 3:
# 1. Đếm số Order ID bị duplicate
# 2. Tạo df_no_duplicates bằng cách giữ bản ghi xuất hiện đầu tiên

duplicate_order_count = df.________(
    subset=["order_id"]
).sum()

print("Duplicate Order IDs:", duplicate_order_count)

df_no_duplicates = df.________(
    subset=["order_id"],
    keep="first"
).copy()

print("Shape before:", df.shape)
print("Shape after :", df_no_duplicates.shape)


# 5. Xác định Numerical và Categorical Variables

### Nhắc lại lý thuyết

Có thể dùng:

```python
df.select_dtypes(include=np.number)
df.select_dtypes(include=["object", "category"])
```

Nhưng cần lưu ý:

> `Customer_ID` hoặc `Order_ID` có thể được lưu dưới dạng số hoặc chuỗi nhưng về bản chất vẫn là **identifier**, không phải numerical feature để tính trung bình.


In [ ]:
# TODO 4:
# Xác định numerical và categorical columns

num_cols = df_no_duplicates.select_dtypes(
    include=________
).columns.tolist()

cat_cols = df_no_duplicates.select_dtypes(
    include=["object", "category"]
).columns.tolist()

print("Numerical columns:", num_cols)
print("Categorical columns:", cat_cols)


# 6. Kiểm tra Unique Values trong Categorical Variables

### Nhắc lại lý thuyết

Các lệnh hữu ích:

```python
nunique()
unique()
value_counts()
```

Chúng giúp phát hiện các nhãn như:

```text
"Hanoi"
"HANOI"
" Hanoi "
"ha noi"
```

đang đại diện cho cùng một nhóm.


In [ ]:
# Ví dụ: xem các giá trị Region
df_no_duplicates["region"].value_counts(dropna=False)


In [ ]:
# TODO 5:
# Kiểm tra unique values của payment_method

print(
    df_no_duplicates["payment_method"].________()
)


# 7. Chuẩn hóa dữ liệu văn bản

### Nhắc lại lý thuyết

Các thao tác thường gặp:

```python
.str.strip()
.str.lower()
.str.upper()
.str.title()
.replace()
```

Quy trình thường là:

```text
loại khoảng trắng
→ chuẩn hóa chữ hoa/thường
→ mapping các nhãn tương đương
```


In [ ]:
# Ví dụ: chuẩn hóa payment_method
df_clean = df_no_duplicates.copy()

df_clean["payment_method"] = (
    df_clean["payment_method"]
    .str.strip()
    .str.lower()
)

df_clean["payment_method"].value_counts(dropna=False)


In [ ]:
# TODO 6:
# Chuẩn hóa Region:
# 1. strip()
# 2. lower()
# 3. map "ha noi" -> "hanoi"
# 4. map "ho chi minh" giữ nguyên
# 5. "da nang" giữ nguyên

df_clean["region"] = (
    df_clean["region"]
    .str.________()
    .str.________()
)

df_clean["region"] = df_clean["region"].replace({
    "________": "hanoi"
})

df_clean["region"].value_counts(dropna=False)


# 8. Chuẩn hóa dữ liệu ngày tháng

### Nhắc lại lý thuyết

Ngày tháng thường được đọc từ CSV dưới dạng `object`.

Có thể dùng:

```python
pd.to_datetime(..., errors="coerce")
```

`errors="coerce"` chuyển giá trị không thể parse thành `NaT`.

Sau đó có thể dùng:

```python
.dt.year
.dt.month
.dt.day
```


In [ ]:
# Xem dữ liệu ngày hiện tại
df_clean["order_date"]


In [ ]:
# TODO 7:
# Chuyển order_date sang datetime
# Gợi ý: format="mixed" hữu ích khi có nhiều định dạng ngày

df_clean["order_date"] = pd.to_datetime(
    df_clean["order_date"],
    format="mixed",
    dayfirst=True,
    errors="________"
)

print(df_clean["order_date"])
print("\nInvalid dates:", df_clean["order_date"].isna().sum())


# 9. Phát hiện Missing Values

### Nhắc lại lý thuyết

Hai hàm tương đương:

```python
df.isnull()
df.isna()
```

Để tính tỷ lệ missing:

```python
df.isna().mean() * 100
```

Lưu ý: missing có thể được mã hóa bằng `"Unknown"`, `-1`, `999`, `"N/A"`... và khi đó Pandas chưa chắc tự nhận diện được.


In [ ]:
# TODO 8:
# Tạo bảng gồm số lượng và tỷ lệ missing theo cột

missing_count = df_clean.________().sum()
missing_percent = df_clean.________().mean() * 100

missing_summary = pd.DataFrame({
    "missing_count": missing_count,
    "missing_percent": missing_percent.round(2)
})

missing_summary.sort_values(
    "missing_percent",
    ascending=False
)


# 10. Xử lý Missing Values

### Nhắc lại lý thuyết

Không có một phương pháp duy nhất.

- Numerical variable:
  - mean;
  - median.
- Categorical variable:
  - mode;
  - `"unknown"`.
- Có thể xóa dòng/cột nếu missing ít hoặc biến không còn hữu ích.

Median thường phù hợp hơn mean nếu biến bị skew hoặc có outliers.


In [ ]:
# Ví dụ: điền payment_method bằng mode
payment_mode = df_clean["payment_method"].mode()[0]

df_clean["payment_method"] = (
    df_clean["payment_method"]
    .fillna(payment_mode)
)

payment_mode


In [ ]:
# TODO 9:
# Điền Unit Price bị thiếu bằng median theo Product.
# Gợi ý: groupby("product")["unit_price"].transform("median")

median_by_product = (
    df_clean
    .groupby("product")["unit_price"]
    .transform("________")
)

df_clean["unit_price"] = (
    df_clean["unit_price"]
    .fillna(________)
)

df_clean[["product", "unit_price"]]


In [ ]:
# TODO 10:
# Với Region bị thiếu, điền bằng mode của Region

region_mode = df_clean["region"].________()[0]
df_clean["region"] = df_clean["region"].fillna(________)

df_clean["region"].value_counts(dropna=False)


# 11. Kiểm tra phạm vi hợp lệ

### Nhắc lại lý thuyết

Một giá trị có thể không missing nhưng vẫn không hợp lệ.

Ví dụ:

```text
Age < 0
Quantity <= 0
Score > 100
```

Có thể dùng:

```python
between()
```

hoặc Boolean filtering.


In [ ]:
# TODO 11:
# Hiển thị các dòng có Quantity <= 0

invalid_quantity = df_clean[
    df_clean["quantity"] ________ 0
]

invalid_quantity


### Câu hỏi

Giá trị `Quantity = -1` có thể có những cách giải thích nào?

- lỗi nhập dữ liệu?
- đơn hàng trả lại?
- điều chỉnh tồn kho?

Không nên tự động xóa trước khi hiểu business rule.


# 12. Kiểm tra Business Rule: Revenue

### Nhắc lại lý thuyết

Một business rule có thể giúp phát hiện dữ liệu sai.

Trong dataset này:

$$
Revenue \approx Quantity \times Unit\ Price
$$

Nếu sai khác lớn, cần điều tra.


In [ ]:
# Tính doanh thu kỳ vọng
df_clean["expected_revenue"] = (
    df_clean["quantity"]
    * df_clean["unit_price"]
)

df_clean[
    [
        "order_id",
        "quantity",
        "unit_price",
        "revenue",
        "expected_revenue"
    ]
]


In [ ]:
# TODO 12:
# Tạo cột revenue_diff = revenue - expected_revenue
# Sau đó hiển thị các dòng có sai khác tuyệt đối > 1

df_clean["revenue_diff"] = (
    df_clean["________"]
    - df_clean["________"]
)

revenue_mismatch = df_clean[
    df_clean["revenue_diff"].abs() > ________
]

revenue_mismatch


# 13. Phát hiện Outlier bằng Boxplot

### Nhắc lại lý thuyết

Boxplot giúp quan sát:

- Q1;
- Median;
- Q3;
- whiskers;
- potential outliers.

Điểm ngoài whiskers **không tự động là dữ liệu sai**.


In [ ]:
plt.figure(figsize=(8, 3))
plt.boxplot(
    df_clean["revenue"].dropna(),
    vert=False
)
plt.xlabel("Revenue")
plt.title("Boxplot of Revenue")
plt.show()


# 14. Phát hiện Outlier bằng IQR

### Nhắc lại lý thuyết

$$
IQR = Q_3 - Q_1
$$

$$
Lower = Q_1 - 1.5IQR
$$

$$
Upper = Q_3 + 1.5IQR
$$

IQR thường ít nhạy với extreme values hơn phương pháp dựa trên mean và standard deviation.


In [ ]:
# TODO 13:
# Tính IQR cho Revenue

q1 = df_clean["revenue"].quantile(________)
q3 = df_clean["revenue"].quantile(________)

iqr = ________ - ________

lower = q1 - 1.5 * ________
upper = q3 + 1.5 * ________

print("Q1:", q1)
print("Q3:", q3)
print("IQR:", iqr)
print("Lower:", lower)
print("Upper:", upper)


In [ ]:
# TODO 14:
# Hiển thị potential outliers

revenue_outliers = df_clean[
    (df_clean["revenue"] < ________)
    |
    (df_clean["revenue"] > ________)
]

revenue_outliers


### Câu hỏi suy nghĩ

1. Có nên xóa ngay `Revenue = 4,000,000` không?
2. Nó có thể là một đơn hàng số lượng lớn hợp lệ không?
3. Nếu đây là dữ liệu fraud detection, việc xóa outlier có thể gây hậu quả gì?


# 15. Chuẩn hóa đơn vị đo

### Nhắc lại lý thuyết

Không thống nhất đơn vị có thể tạo ra outlier giả.

Ví dụ:

```text
1.2 kg
800 g
1.5 kg
650 g
```

Cần chuyển về cùng một đơn vị trước khi phân tích.


In [ ]:
# Xem dữ liệu trọng lượng hiện tại
df_clean[["weight", "weight_unit"]]


In [ ]:
# TODO 15:
# Chuyển toàn bộ trọng lượng về kg.
# Nếu weight_unit == "g" thì chia weight cho 1000.
# Sau đó đặt toàn bộ weight_unit thành "kg".

mask_g = df_clean["weight_unit"] == "g"

df_clean.loc[
    mask_g,
    "weight"
] = (
    df_clean.loc[
        mask_g,
        "weight"
    ] / ________
)

df_clean["weight_unit"] = "________"

df_clean[["weight", "weight_unit"]]


# 16. Data Validation sau Cleaning

### Nhắc lại lý thuyết

Sau khi làm sạch cần kiểm tra lại:

```text
missing?
duplicate?
invalid range?
wrong dtype?
unexpected category?
business rule?
```

Assertions giúp tự động phát hiện lỗi:

```python
assert condition
```


In [ ]:
# Một số kiểm tra tổng quát
print("Duplicate Order IDs:",
      df_clean.duplicated(subset=["order_id"]).sum())

print("\nMissing values:")
print(df_clean.isna().sum())

print("\nData types:")
print(df_clean.dtypes)


In [ ]:
# TODO 16:
# Hoàn thiện các assertions hợp lý

assert df_clean.duplicated(
    subset=["order_id"]
).sum() == ________

assert (df_clean["unit_price"] > 0).________()

assert df_clean["region"].isna().sum() == ________

print("Các kiểm tra validation cơ bản đã chạy.")


# 17. Tổng kết dữ liệu sau Cleaning

Hãy xem lại dataset:


In [ ]:
df_clean


In [ ]:
# TODO 17:
# Chạy thống kê mô tả cho numerical variables
df_clean.________()


# 18. Bài tập tổng hợp

Không xem lại các ví dụ phía trên trong lần làm đầu tiên nếu có thể.

## Yêu cầu

Từ `df` ban đầu, hãy tạo một DataFrame mới tên:

```python
final_df
```

và thực hiện toàn bộ quy trình:

1. Chuẩn hóa tên cột.
2. Loại duplicate `order_id`.
3. Chuẩn hóa `region`.
4. Chuẩn hóa `payment_method`.
5. Chuyển `order_date` sang datetime.
6. Xử lý missing `unit_price`.
7. Xử lý missing `region`.
8. Xử lý missing `payment_method`.
9. Kiểm tra `quantity <= 0`.
10. Chuyển trọng lượng về kg.
11. Tính `expected_revenue`.
12. Tìm các dòng vi phạm business rule.
13. Tính IQR của `revenue`.
14. Liệt kê potential outliers.
15. Validation lại dữ liệu.

> **Yêu cầu quan trọng:** Không tự động xóa outliers nếu chưa có lý do nghiệp vụ.


In [ ]:
# TODO 18 - Bài tập tổng hợp
# Viết toàn bộ quy trình của bạn ở đây.

final_df = df.copy()

# 1. Chuẩn hóa tên cột
# ...

# 2. Loại duplicate order_id
# ...

# 3. Chuẩn hóa region
# ...

# 4. Chuẩn hóa payment_method
# ...

# 5. Convert order_date
# ...

# 6-8. Missing values
# ...

# 9. Invalid quantity
# ...

# 10. Weight -> kg
# ...

# 11-12. Business rule
# ...

# 13-14. Outlier
# ...

# 15. Validation
# ...

final_df.head()


# 19. Câu hỏi tự kiểm tra

Trả lời bằng Markdown ngay dưới mỗi câu.

1. Data Cleaning khác Data Preprocessing như thế nào?
2. Vì sao không nên gọi `drop_duplicates()` một cách máy móc?
3. Tại sao `Customer_ID` không nên được xem như numerical feature thông thường?
4. `isna()` và `isnull()` khác nhau không?
5. Vì sao median có thể tốt hơn mean khi dữ liệu có outliers?
6. Tại sao một outlier không nhất thiết là lỗi?
7. Vì sao cần chuẩn hóa đơn vị trước khi phát hiện outliers?
8. `errors="coerce"` trong `pd.to_datetime()` làm gì?
9. Business rule có vai trò gì trong Data Validation?
10. Sau Data Cleaning, tại sao vẫn cần validation lại?


# 20. Mở rộng

Nếu đã hoàn thành phần chính, hãy thử:

### Bài mở rộng A — Capping Outlier

Thay vì xóa outlier của `revenue`, dùng:

```python
Series.clip()
```

để giới hạn revenue trong `[lower, upper]`.

### Bài mở rộng B — Hàm làm sạch văn bản

Viết hàm:

```python
def clean_text_column(series):
    ...
```

để:

- strip;
- lower;
- chuẩn hóa khoảng trắng.

### Bài mở rộng C — Tự động hóa validation

Viết hàm:

```python
def validate_sales_data(df):
    ...
```

trả về một dictionary như:

```python
{
    "duplicate_order_id": ...,
    "missing_unit_price": ...,
    "invalid_quantity": ...,
    "revenue_mismatch": ...
}
```


# 21. Checklist trước khi kết thúc

```text
□ Tôi đã kiểm tra shape và info()
□ Tôi đã kiểm tra duplicates
□ Tôi đã phân biệt numerical/categorical variables
□ Tôi đã kiểm tra unique values
□ Tôi đã chuẩn hóa text
□ Tôi đã chuẩn hóa tên cột
□ Tôi đã chuyển date sang datetime
□ Tôi đã kiểm tra missing values
□ Tôi đã chọn cách imputation phù hợp
□ Tôi đã kiểm tra invalid values
□ Tôi đã kiểm tra business rules
□ Tôi đã kiểm tra potential outliers
□ Tôi không tự động xóa outliers
□ Tôi đã chuẩn hóa đơn vị
□ Tôi đã validation lại dataset
```
